# reconcile

> Cloud-init fragment for the reconcile-web viewer: dedicated system user, venv install, archive dir, hardened systemd unit

In [ ]:
#| default_exp reconcile

In [ ]:
#| hide
from nbdev.showdoc import *

Read-only viewer over the bookkeeping archive — the one fragment that must
NOT mirror discopipe's write grants (panel-adjudicated, 2026-07-17):

- **Archive is pushed, not pulled.** The laptop's `~/reconcile-archive` is the
  source of truth; `rsync` pushes it to `/srv/reconcile-archive`
  (`doyu:reconcile`, setgid 2750). The service user reads via group only —
  no deploy keys, no git on the box.
- **No write path.** `ProtectSystem=strict` with no `ReadWritePaths`, plus
  `ProtectHome=yes` (nothing under `/home` is needed — discopipe could not
  afford that, this unit can).
- Secrets (`ARCHIVE_DIR`, `APP_PASSWORD`, `SESSION_SECRET`) live in
  `/etc/reconcile/env`, installed post-boot; until then the unit stays
  inactive via `ConditionPathExists`. Post-boot order: rsync archive →
  install env → `systemctl start reconcile`.

## `reconcile_service`

In [ ]:
#| export
from boxrecipe.services import check_service, write_file_cmd

_UNIT = """[Unit]
Description=reconcile - read-only web viewer over the reconcile-archive bookkeeping data
After=network-online.target
Wants=network-online.target
ConditionPathExists=/etc/reconcile/env
ConditionPathIsDirectory=/srv/reconcile-archive
StartLimitIntervalSec=60
StartLimitBurst=3

[Service]
User=reconcile
EnvironmentFile=/etc/reconcile/env
ExecStart=/opt/reconcile/bin/uvicorn --factory reconcile_web.app:create_app --host 127.0.0.1 --port 5001
Restart=on-failure
RestartSec=5
NoNewPrivileges=yes
ProtectSystem=strict
ProtectHome=yes
PrivateTmp=yes

[Install]
WantedBy=multi-user.target
"""

def reconcile_service(
)->dict:  # validated service dict for the reconcile-web viewer (public site + install + unit cmds)
    """Service dict for reconcile-web: read-only bookkeeping viewer, app-password gated."""
    cmds = [
        "useradd --system --no-create-home --shell /usr/sbin/nologin reconcile",
        # doyu joins group reconcile: without membership, the kernel silently
        # clears the setgid bit on rsync's --chmod=D2750 dirs, so pushed files
        # fall back to group doyu and the service user cannot read the archive
        "usermod -aG reconcile doyu",
        "python3 -m venv /opt/reconcile",
        "/opt/reconcile/bin/pip install git+https://github.com/doyu/reconcile-web.git",
        # group-read for the service user, nothing for others; X keeps the venv
        # executable whatever umask the surrounding cloud-init script runs under
        "chown -R root:reconcile /opt/reconcile",
        "chmod -R u=rwX,g=rX,o= /opt/reconcile",
        # rsync target: doyu pushes from the laptop, setgid keeps group reconcile
        "install -d -m 2750 -o doyu -g reconcile /srv/reconcile-archive",
        "install -d -m 700 -o root -g root /etc/reconcile",
        write_file_cmd("/etc/systemd/system/reconcile.service", _UNIT),
        "systemctl daemon-reload",
        "systemctl enable reconcile",
    ]
    # public=True: the accountant has no VPN — APP_PASSWORD is the gate
    return check_service({"name": "reconcile", "domain": "reconcile.ninjalabo.ai",
                          "port": 5001, "public": True,
                          "packages": ["python3-venv"], "cmds": cmds})

In [ ]:
svc = reconcile_service()
assert svc["name"] == "reconcile" and svc["domain"] == "reconcile.ninjalabo.ai"
# public: the accountant reaches it from the open internet (app password is
# the only gate — the maturity-path flag flipped 2026-07-17)
assert svc["port"] == 5001 and svc["public"] is True
assert "python3-venv" in svc["packages"]

joined = "\n".join(svc["cmds"])
# system user, no home: the viewer owns nothing on disk
assert "useradd --system --no-create-home --shell /usr/sbin/nologin reconcile" in joined
# doyu must join group reconcile: a non-member's rsync --chmod=D2750 gets its
# setgid bit silently cleared by the kernel, so pushed files fall back to
# group doyu and the service user cannot read them (2026-08-21 outage)
assert "usermod -aG reconcile doyu" in joined
assert joined.index("useradd --system") < joined.index("usermod -aG reconcile doyu")
# venv install from the (public) app repo
assert "python3 -m venv /opt/reconcile" in joined
assert "/opt/reconcile/bin/pip install git+https://github.com/doyu/reconcile-web.git" in joined
# service user reads the venv via group; other users get nothing
assert "chown -R root:reconcile /opt/reconcile" in joined
assert "chmod -R u=rwX,g=rX,o= /opt/reconcile" in joined
assert joined.index("pip install") < joined.index("chmod -R u=rwX")
# archive dir: doyu rsyncs into it, group reconcile reads, setgid keeps the group
assert "install -d -m 2750 -o doyu -g reconcile /srv/reconcile-archive" in joined
# secrets dir (root-only) so the post-boot `install -m 600` has a target
assert "install -d -m 700 -o root -g root /etc/reconcile" in joined
# the unit: identity, wait-state, network ordering, hardening, restart policy
for token in ("/etc/systemd/system/reconcile.service",
              "User=reconcile",
              "EnvironmentFile=/etc/reconcile/env",
              "ConditionPathExists=/etc/reconcile/env",
              "ConditionPathIsDirectory=/srv/reconcile-archive",
              "After=network-online.target", "Wants=network-online.target",
              "ExecStart=/opt/reconcile/bin/uvicorn --factory reconcile_web.app:create_app --host 127.0.0.1 --port 5001",
              "Restart=on-failure", "RestartSec=5",
              "StartLimitIntervalSec=60", "StartLimitBurst=3",
              "NoNewPrivileges=yes", "ProtectSystem=strict",
              "ProtectHome=yes", "PrivateTmp=yes",
              "WantedBy=multi-user.target"):
    assert token in joined, token
# read-only viewer: never grant a write path (do not mirror discopipe here)
assert "ReadWritePaths" not in joined
assert "systemctl daemon-reload" in joined
assert "systemctl enable reconcile" in joined
# never a secret name/value in the fragment — env values arrive post-boot
for name in ("APP_PASSWORD", "SESSION_SECRET", "ARCHIVE_DIR"):
    assert name not in joined, name

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()